# Automatic Customer Complaint Triage (Urgency & Department)

**Business problem:** every incoming support ticket has to be routed to the right team (**Billing**, **Technical Support**, **Sales**) and prioritized (**High**, **Medium**, **Low**). Doing this by hand is slow, and a misrouted *High* ticket costs money and customers.

We compare two approaches on 75 labeled tickets:

| | Model 1 | Model 2 |
|---|---|---|
| Type | Open source, runs locally | Proprietary API |
| Model | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (zero-shot NLI) | `gpt-4o-mini` (temperature 0, structured JSON output) |

**Before running:**
1. `Runtime > Change runtime type > T4 GPU` (works on CPU too, just slower).
2. Upload `tickets_clients.csv` to the Colab file panel (the notebook will ask for it if missing).
3. Add your OpenAI key in the 🔑 **Secrets** panel as `OPENAI_API_KEY` and enable notebook access (otherwise you'll be prompted for it).

In [ ]:
# Cell 1 - Install dependencies (transformers, torch, pandas and sklearn are preinstalled in Colab)
!pip install -q openai sentencepiece protobuf

In [ ]:
# Cell 2 - Imports and global configuration
import os
import json
import time

import pandas as pd
import torch
from IPython.display import display
from openai import OpenAI
from sklearn.metrics import accuracy_score, recall_score
from transformers import pipeline

# ---- Data ----
CSV_PATH = "tickets_clients.csv"
TEXT_COL = "text"
URGENCY_COL = "true_urgency"
DEPARTMENT_COL = "true_department"

URGENCY_LABELS = ["High", "Medium", "Low"]
DEPARTMENT_LABELS = ["Billing", "Technical Support", "Sales"]
CRITICAL_CLASS = "High"  # the class we must not miss

# ---- Models ----
HF_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
OPENAI_MODEL = "gpt-4o-mini"

# ---- Cost assumptions (USD) - adjust to your provider's current prices ----
OPENAI_INPUT_PRICE_PER_1M = 0.15    # gpt-4o-mini, per 1M input tokens
OPENAI_OUTPUT_PRICE_PER_1M = 0.60   # gpt-4o-mini, per 1M output tokens
GPU_PRICE_PER_HOUR = 0.35           # on-demand T4-class cloud GPU
CPU_PRICE_PER_HOUR = 0.10           # small cloud CPU instance

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [ ]:
# Cell 3 - Load and validate the dataset
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print(f"'{CSV_PATH}' not found. Please upload it:")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Place '{CSV_PATH}' next to this notebook.")

df = pd.read_csv(CSV_PATH)

# Basic schema checks so we fail early with a clear message
missing_cols = {TEXT_COL, URGENCY_COL, DEPARTMENT_COL} - set(df.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"

for col in [TEXT_COL, URGENCY_COL, DEPARTMENT_COL]:
    df[col] = df[col].astype(str).str.strip()

assert set(df[URGENCY_COL]) <= set(URGENCY_LABELS), f"Unexpected urgency labels: {set(df[URGENCY_COL])}"
assert set(df[DEPARTMENT_COL]) <= set(DEPARTMENT_LABELS), f"Unexpected department labels: {set(df[DEPARTMENT_COL])}"

print(f"Loaded {len(df)} tickets\n")
print(pd.crosstab(df[URGENCY_COL], df[DEPARTMENT_COL], margins=True))
display(df.head())

Loaded 75 tickets

true_department  Billing  Sales  Technical Support  All
true_urgency                                           
High                   8      8                  9   25
Low                    8      9                  8   25
Medium                 9      8                  8   25
All                   25     25                 25   75


,text,true_urgency,true_department
0,Can I change the company name that appears on our invoices? We rebranded recently.,Low,Billing
1,What's the difference between the Pro and Business plans? We mostly care about the reporting features.,Medium,Sales
2,We were charged in USD instead of EUR this time. Can you switch the currency for future invoices and adjust the last one?,Medium,Billing
3,Could you send me a brochure or some case studies to share with my manager next quarter?,Low,Sales
4,"Reports load really slowly in the afternoons, sometimes 2 or 3 minutes. Is there a known issue?",Medium,Technical Support


## Model 1 - Hugging Face zero-shot classification (open source)

Zero-shot NLI models score how well each candidate label "entails" the text. Short descriptive labels work better than single words, so each class gets a natural-language description. The description is then mapped back to the class name.

In [ ]:
# Cell 4 - Load the zero-shot pipeline (loading time is reported separately from inference time)
device = 0 if torch.cuda.is_available() else -1
print("Device:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

t0 = time.perf_counter()
zero_shot = pipeline("zero-shot-classification", model=HF_MODEL, device=device)
print(f"Model loaded in {time.perf_counter() - t0:.1f} s")

HYPOTHESIS_TEMPLATE = "This customer message is {}."

# Descriptive candidate labels -> class name
URGENCY_DESCRIPTIONS = {
    "High": "an emergency that is blocking the business, losing money or a security risk and needs action today",
    "Medium": "a problem or request that should be handled soon but is not an emergency",
    "Low": "a simple question, suggestion or minor detail with no urgency",
}
DEPARTMENT_DESCRIPTIONS = {
    "Billing": "about billing, payments, charges, refunds or invoices",
    "Technical Support": "about a technical problem, bug, error, outage or how to use the software",
    "Sales": "about buying, pricing, plans, quotes, licenses, demos or discounts",
}
_URG_INV = {v: k for k, v in URGENCY_DESCRIPTIONS.items()}
_DEP_INV = {v: k for k, v in DEPARTMENT_DESCRIPTIONS.items()}


def hf_classify(texts, batch_size=8):
    # Return a list of {'urgency', 'department', 'urgency_score', 'department_score'} for each text.
    urg_out = zero_shot(texts, candidate_labels=list(URGENCY_DESCRIPTIONS.values()),
                        hypothesis_template=HYPOTHESIS_TEMPLATE, batch_size=batch_size)
    dep_out = zero_shot(texts, candidate_labels=list(DEPARTMENT_DESCRIPTIONS.values()),
                        hypothesis_template=HYPOTHESIS_TEMPLATE, batch_size=batch_size)
    # The pipeline returns a dict (not a list) when given a single text
    if isinstance(urg_out, dict):
        urg_out, dep_out = [urg_out], [dep_out]
    return [
        {
            "urgency": _URG_INV[u["labels"][0]],
            "department": _DEP_INV[d["labels"][0]],
            "urgency_score": u["scores"][0],
            "department_score": d["scores"][0],
        }
        for u, d in zip(urg_out, dep_out)
    ]


# Warm-up call so the first real prediction doesn't pay CUDA/kernel initialization costs
_ = hf_classify(["Warm-up message."])
print("Warm-up done")

Device: Tesla T4


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Model loaded in 11.1 s
Warm-up done


In [ ]:
# Cell 5 - Run Model 1 on all tickets and time it
t0 = time.perf_counter()
hf_results = hf_classify(df[TEXT_COL].tolist())
hf_total_time = time.perf_counter() - t0

df["hf_urgency"] = [r["urgency"] for r in hf_results]
df["hf_department"] = [r["department"] for r in hf_results]
hf_latency = hf_total_time / len(df)

print(f"Hugging Face total time: {hf_total_time:.2f} s  |  avg per ticket: {hf_latency:.4f} s")
display(df[[TEXT_COL, URGENCY_COL, "hf_urgency", DEPARTMENT_COL, "hf_department"]].head())

Hugging Face total time: 2.16 s  |  avg per ticket: 0.0287 s


,text,true_urgency,hf_urgency,true_department,hf_department
0,Can I change the company name that appears on our invoices? We rebranded recently.,Low,Low,Billing,Billing
1,What's the difference between the Pro and Business plans? We mostly care about the reporting features.,Medium,Medium,Sales,Billing
2,We were charged in USD instead of EUR this time. Can you switch the currency for future invoices and adjust the last one?,Medium,Low,Billing,Billing
3,Could you send me a brochure or some case studies to share with my manager next quarter?,Low,Low,Sales,Billing
4,"Reports load really slowly in the afternoons, sometimes 2 or 3 minutes. Is there a known issue?",Medium,Low,Technical Support,Billing


## Model 2 - OpenAI `gpt-4o-mini` (proprietary)

One API call per ticket returns **both** labels. **Structured Outputs** (a JSON schema with `enum` values) guarantee valid JSON that only contains our exact labels, so there is no fragile text parsing.

In [ ]:
# Cell 6 - OpenAI client (key from Colab Secrets, environment variable, or prompt)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=api_key)

SYSTEM_PROMPT = '''You are a customer support triage assistant for a software / e-commerce company.
Classify the customer's message on two axes.

urgency:
- High: business is blocked, money is being lost, security risk, wrong/duplicate charges, or a hard deadline today/tomorrow.
- Medium: a real problem or request that should be handled soon, but there is a workaround or no immediate damage.
- Low: general questions, suggestions, cosmetic issues, or future plans with no time pressure.

department:
- Billing: charges, refunds, invoices, payment methods, billing cycles, taxes.
- Technical Support: bugs, outages, errors, integrations, security/access, how to use the product.
- Sales: buying, pricing, plans, quotes, licenses, demos, trials, discounts, partnerships.

Watch for sarcasm and mixed messages: judge by the actual problem, not the polite or ironic wording.'''

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "ticket_triage",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "urgency": {"type": "string", "enum": URGENCY_LABELS},
                "department": {"type": "string", "enum": DEPARTMENT_LABELS},
            },
            "required": ["urgency", "department"],
            "additionalProperties": False,
        },
    },
}


def openai_classify(text, max_retries=3):
    # Classify one ticket. Returns labels, latency (s) and token usage. Retries on transient errors.
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},   #instructions
                    {"role": "user", "content": text},  #ticket to classify
                ],
                response_format=RESPONSE_FORMAT,
            )
            latency = time.perf_counter() - t0
            data = json.loads(resp.choices[0].message.content)
            return {
                "urgency": data["urgency"],
                "department": data["department"],
                "latency": latency,
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
            }
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  Error: {e}. Retrying in {wait} s...")
            time.sleep(wait)

Enter your OpenAI API key: ··········


In [ ]:
# Cell 7 - Run Model 2 on all tickets and time it
oai_results = []
t0 = time.perf_counter()
for i, text in enumerate(df[TEXT_COL], start=1):
    oai_results.append(openai_classify(text))
    if i % 15 == 0 or i == len(df):
        print(f"  {i}/{len(df)} tickets classified")
oai_total_time = time.perf_counter() - t0

df["oai_urgency"] = [r["urgency"] for r in oai_results]
df["oai_department"] = [r["department"] for r in oai_results]
oai_latency = oai_total_time / len(df)

oai_avg_in = sum(r["input_tokens"] for r in oai_results) / len(oai_results)
oai_avg_out = sum(r["output_tokens"] for r in oai_results) / len(oai_results)

print(f"\nOpenAI total time: {oai_total_time:.2f} s  |  avg per ticket: {oai_latency:.4f} s")
print(f"Avg tokens per ticket: {oai_avg_in:.0f} input / {oai_avg_out:.0f} output")

  15/75 tickets classified
  30/75 tickets classified
  45/75 tickets classified
  60/75 tickets classified
  75/75 tickets classified

OpenAI total time: 62.78 s  |  avg per ticket: 0.8370 s
Avg tokens per ticket: 269 input / 10 output


## Evaluation

**Why recall on "High" is the critical metric.** A *High* ticket predicted as Medium/Low sits in the queue while the customer loses money, gets double-charged, or churns. A false alarm (a Low ticket marked High) only costs a few minutes of an agent's time. Missing urgent tickets is far more expensive than over-prioritizing, so **recall on High** (the share of truly urgent tickets we catch) matters more than overall accuracy.

**Cost assumptions:** OpenAI cost is based on the real token usage measured above × list prices. Hugging Face cost is the compute time per ticket × the hourly price of the hardware it ran on (it has no per-call fee, but someone pays for the server).

In [ ]:
# Cell 8 - Comparison table
def pct(x):
    return round(100 * x, 1)


def critical_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, labels=[CRITICAL_CLASS], average=None, zero_division=0)[0]


hw_price = GPU_PRICE_PER_HOUR if device == 0 else CPU_PRICE_PER_HOUR
hf_cost_10k = hf_latency * 10_000 / 3600 * hw_price
oai_cost_10k = 10_000 * (oai_avg_in * OPENAI_INPUT_PRICE_PER_1M + oai_avg_out * OPENAI_OUTPUT_PRICE_PER_1M) / 1_000_000

y_urg, y_dep = df[URGENCY_COL], df[DEPARTMENT_COL]

comparison = pd.DataFrame(
    {
        "Hugging Face (mDeBERTa zero-shot)": [
            pct(accuracy_score(y_urg, df["hf_urgency"])),
            pct(accuracy_score(y_dep, df["hf_department"])),
            pct(critical_recall(y_urg, df["hf_urgency"])),
            round(hf_latency, 4),
            round(hf_cost_10k, 4),
        ],
        "OpenAI (gpt-4o-mini)": [
            pct(accuracy_score(y_urg, df["oai_urgency"])),
            pct(accuracy_score(y_dep, df["oai_department"])),
            pct(critical_recall(y_urg, df["oai_urgency"])),
            round(oai_latency, 4),
            round(oai_cost_10k, 4),
        ],
    },
    index=[
        "Urgency accuracy (%)",
        "Department accuracy (%)",
        f"Recall '{CRITICAL_CLASS}' urgency (%)  <- critical",
        "Avg latency per ticket (s)",
        "Estimated cost per 10,000 tickets (USD)",
    ],
)
comparison.index.name = "Metric"
display(comparison)
print(f"HF cost assumes {'GPU' if device == 0 else 'CPU'} at ${hw_price}/h; "
      f"OpenAI cost assumes ${OPENAI_INPUT_PRICE_PER_1M}/1M input and ${OPENAI_OUTPUT_PRICE_PER_1M}/1M output tokens.")

,Hugging Face (mDeBERTa zero-shot),OpenAI (gpt-4o-mini)
Metric,,
Urgency accuracy (%),56.0000,84.0000
Department accuracy (%),64.0000,96.0000
Recall 'High' urgency (%) <- critical,36.0000,100.0000
Avg latency per ticket (s),0.0287,0.8370
"Estimated cost per 10,000 tickets (USD)",0.0279,0.4648


HF cost assumes GPU at $0.35/h; OpenAI cost assumes $0.15/1M input and $0.6/1M output tokens.


In [ ]:
# Cell 9 - Confusion matrices (rows = true label, columns = predicted label)
for name, prefix in [("Hugging Face", "hf"), ("OpenAI", "oai")]:
    print(f"\n=== {name} - Urgency ===")
    display(pd.crosstab(df[URGENCY_COL], df[f"{prefix}_urgency"], rownames=["true"], colnames=["predicted"])
            .reindex(index=URGENCY_LABELS, columns=URGENCY_LABELS, fill_value=0))
    print(f"=== {name} - Department ===")
    display(pd.crosstab(df[DEPARTMENT_COL], df[f"{prefix}_department"], rownames=["true"], colnames=["predicted"])
            .reindex(index=DEPARTMENT_LABELS, columns=DEPARTMENT_LABELS, fill_value=0))


=== Hugging Face - Urgency ===


predicted,High,Medium,Low
true,,,
High,9,15,1
Medium,1,10,14
Low,0,2,23


=== Hugging Face - Department ===


predicted,Billing,Technical Support,Sales
true,,,
Billing,24,1,0
Technical Support,7,16,2
Sales,12,5,8



=== OpenAI - Urgency ===


predicted,High,Medium,Low
true,,,
High,25,0,0
Medium,2,14,9
Low,0,1,24


=== OpenAI - Department ===


predicted,Billing,Technical Support,Sales
true,,,
Billing,23,0,2
Technical Support,0,24,1
Sales,0,0,25


## Error analysis

Only the tickets where **at least one model** made a mistake. Wrong predictions are highlighted in red.

In [ ]:
# Cell 10 - Tickets where at least one model failed
hf_wrong = (df["hf_urgency"] != df[URGENCY_COL]) | (df["hf_department"] != df[DEPARTMENT_COL])
oai_wrong = (df["oai_urgency"] != df[URGENCY_COL]) | (df["oai_department"] != df[DEPARTMENT_COL])

error_df = df.loc[hf_wrong | oai_wrong, [
    TEXT_COL,
    URGENCY_COL, "hf_urgency", "oai_urgency",
    DEPARTMENT_COL, "hf_department", "oai_department",
]].copy()
error_df.insert(1, "failed_by", [
    "Both" if h and o else ("Hugging Face" if h else "OpenAI")
    for h, o in zip(hf_wrong[error_df.index], oai_wrong[error_df.index])
])

print(f"Tickets with at least one error: {len(error_df)} / {len(df)}")
print(error_df["failed_by"].value_counts().to_string(), "\n")

PAIRS = [("hf_urgency", URGENCY_COL), ("oai_urgency", URGENCY_COL),
         ("hf_department", DEPARTMENT_COL), ("oai_department", DEPARTMENT_COL)]


def highlight_errors(frame):
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    for pred_col, true_col in PAIRS:
        styles.loc[frame[pred_col] != frame[true_col], pred_col] = "background-color: #f8d7da; color: #721c24"
    return styles


display(error_df.style.apply(highlight_errors, axis=None))

Tickets with at least one error: 54 / 75
failed_by
Hugging Face    40
Both            11
OpenAI           3 



,text,failed_by,true_urgency,hf_urgency,oai_urgency,true_department,hf_department,oai_department
0,Can I change the company name that appears on our invoices? We rebranded recently.,OpenAI,Low,Low,Medium,Billing,Billing,Billing
1,What's the difference between the Pro and Business plans? We mostly care about the reporting features.,Both,Medium,Medium,Low,Sales,Billing,Sales
2,We were charged in USD instead of EUR this time. Can you switch the currency for future invoices and adjust the last one?,Hugging Face,Medium,Low,Medium,Billing,Billing,Billing
3,Could you send me a brochure or some case studies to share with my manager next quarter?,Hugging Face,Low,Low,Low,Sales,Billing,Sales
4,"Reports load really slowly in the afternoons, sometimes 2 or 3 minutes. Is there a known issue?",Hugging Face,Medium,Low,Medium,Technical Support,Billing,Technical Support
5,"Your rep told us the volume discount was locked in, and now the renewal quote is 40% higher. Our contract expires tomorrow, so sort this out or we walk.",Hugging Face,High,Medium,High,Sales,Sales,Sales
8,Our board meets tomorrow morning to approve the vendor. I need Enterprise tier pricing and a signed NDA before then.,Hugging Face,High,Medium,High,Sales,Sales,Sales
10,Small typo on the settings page: it says 'Notifcations' instead of 'Notifications'.,Hugging Face,Low,Low,Low,Technical Support,Billing,Technical Support
12,"Do you offer a free plan for very small shops, like a one-person business?",Hugging Face,Low,Low,Low,Sales,Billing,Sales
13,Some of my product descriptions show weird characters instead of accents after the bulk import.,Hugging Face,Medium,Medium,Medium,Technical Support,Sales,Technical Support


## Live demo

Type a made-up complaint and both models classify it right away. Press **Enter** on an empty line (or type `exit`) to stop.

In [ ]:
# Cell 11 - Interactive live demo
print("Live triage demo - type a customer complaint (empty line or 'exit' to stop)")

while True:
    complaint = input("\nCustomer message > ").strip()
    if complaint.lower() in {"", "exit", "quit"}:
        print("Demo finished.")
        break

    t0 = time.perf_counter()
    hf_pred = hf_classify([complaint], batch_size=1)[0]
    hf_time = time.perf_counter() - t0

    oai_pred = openai_classify(complaint)

    print("-" * 70)
    print(f"Hugging Face | Urgency: {hf_pred['urgency']:<7} ({hf_pred['urgency_score']:.0%})"
          f" | Department: {hf_pred['department']:<18} ({hf_pred['department_score']:.0%})"
          f" | {hf_time:.3f} s")
    print(f"OpenAI       | Urgency: {oai_pred['urgency']:<7}       "
          f" | Department: {oai_pred['department']:<18}       "
          f" | {oai_pred['latency']:.3f} s")
    print("-" * 70)

Live triage demo - type a customer complaint (empty line or 'exit' to stop)

Customer message > My user credentials don´t work
----------------------------------------------------------------------
Hugging Face | Urgency: Low     (52%) | Department: Technical Support  (39%) | 0.162 s
OpenAI       | Urgency: High           | Department: Technical Support         | 0.812 s
----------------------------------------------------------------------

Customer message > My user credentials don´t work
----------------------------------------------------------------------
Hugging Face | Urgency: Low     (52%) | Department: Technical Support  (39%) | 0.196 s
OpenAI       | Urgency: High           | Department: Technical Support         | 0.857 s
----------------------------------------------------------------------


KeyboardInterrupt: Interrupted by user